# 머신러닝 전처리
- 데이터 전처리 방법과 같음
- 데이터 누수로 인해 순서가 다름  
-> 파일 로드  
-> Defects 칼럼 유형별 정리(target 설정)  
-> 식별자 생성  
-> 데이터 시간순으로 정렬  
-> train / test 분리  
-> 단일값 삭제  
-> 결측치  
-> 이상치  

In [3]:
# 데이터 처리 및 분석
import os
import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import warnings
import platform

# 시각화
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

# 한글 폰트 설정
if platform.system() == 'Windows':
    plt.rcParams['font.family'] = 'Malgun Gothic'
elif platform.system() == 'Darwin':
    plt.rcParams['font.family'] = 'AppleGothic'
else:
    plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False

In [4]:
# 데이터 로드
df = pd.read_csv("DieCasting_Quality_Raw_Data.csv", header=[0,1])

In [7]:
df.head()

Process                                                                   \
       id Product_Type Shot Velocity_1 Velocity_2 Velocity_3 High_Velocity   
0       1            1    1      0.144      0.170      0.188         2.134   
1    1002            1    2      0.144      0.170      0.182         2.124   
2    2003            1    3      0.144      0.170      0.182         2.116   
3    3004            1    4      0.144      0.170      0.182         2.137   
4    4005            1    5      0.144      0.172      0.176         2.111   

                                                                        \
  Cylinder_Pressure Rapid_Rise_Time Biscuit_Thickness  Clamping_Force    
0               214           0.008                 10             258   
1               217           0.008                 11             257   
2               214           0.008                 11             257   
3               217           0.008                 11             257   
4               217           0.008                 12             257   

                                                                           \
  Cycle_Time  Pressure_Rise_Time Casting_Pressure Spray_Time Spray_1_Time   
0       20.7               0.044             1037        7.8          0.7   
1       20.7               0.044             1052        7.8          0.7   
2       20.8               0.041             1037        7.8          0.7   
3       20.7               0.043             1051        7.8          0.7   
4       20.7               0.042             1052        7.8          0.7   

                             Sensor                                \
  Spray_2_Time Melting_Furnace_Temp Air_Pressure Air_Pressure_Min   
0          0.8                695.0          6.3                3   
1          0.8                696.4          6.3                3   
2          0.8                696.4          6.3                3   
3          0.8                696.4          6.3                3   
4          0.8                697.9          6.4                3   

                                                                   \
  Air_Pressure_Max Coolant_Temp Coolant_Temp_Min Coolant_Temp_Max   
0                9         26.0               10               50   
1                9         26.1               10               50   
2                9         26.1               10               50   
3                9         26.1               10               50   
4                9         26.1               10               50   

                                                                   \
  Coolant_Pressure Factory_Temp Factory_Temp_Min Factory_Temp_Max   
0             2.71         32.9             18.0             22.0   
1             2.69         32.9             18.0             22.0   
2             2.69         32.9             18.0             22.0   
3             2.69         32.9             18.0             22.0   
4             2.69         32.9             18.0             22.0   

                                                                  Defects  \
  Factory_Humidity Factory_Humidity_Min Factory_Humidity_Max Short_Shot_1   
0             58.4                 18.0                 22.0            0   
1             58.2                 18.0                 22.0            0   
2             58.2                 18.0                 22.0            0   
3             58.2                 18.0                 22.0            0   
4             57.8                 18.0                 22.0            0   

                                                                   \
  Bubble_1 Exfoliation_1 Blow_Hole_1 Stain_1 Dent_1 Deformation_1   
0        0             0           0       0      0             0   
1        0             0           0       0      0             0   
2        0             0           0       0      0             0   
3        0             1           0       0      0        

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7535 entries, 0 to 7534
Data columns (total 57 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   (Process, id)                   7535 non-null   int64  
 1   (Process, Product_Type)         7535 non-null   int64  
 2   (Process, Shot)                 7535 non-null   int64  
 3   (Process, Velocity_1)           7535 non-null   float64
 4   (Process, Velocity_2)           7535 non-null   float64
 5   (Process, Velocity_3)           7535 non-null   float64
 6   (Process, High_Velocity)        7535 non-null   float64
 7   (Process, Cylinder_Pressure)    7535 non-null   int64  
 8   (Process, Rapid_Rise_Time)      7535 non-null   float64
 9   (Process, Biscuit_Thickness )   7535 non-null   int64  
 10  (Process, Clamping_Force )      7535 non-null   int64  
 11  (Process, Cycle_Time)           7535 non-null   float64
 12  (Process,  Pressure_Rise_Time)  75

In [12]:
display(df.isnull().sum().sort_values(ascending=False).head(20))

Sensor   Factory_Temp            90
         Factory_Humidity        90
         Factory_Temp_Max        90
         Factory_Humidity_Max    90
         Factory_Humidity_Min    90
         Factory_Temp_Min        90
Process  Shot                     0
         Product_Type             0
         id                       0
         Biscuit_Thickness        0
         Clamping_Force           0
         Cycle_Time               0
          Pressure_Rise_Time      0
         Casting_Pressure         0
         Spray_Time               0
         Spray_1_Time             0
         Spray_2_Time             0
Sensor   Melting_Furnace_Temp     0
         Air_Pressure             0
Process  Velocity_1               0
dtype: int64

In [13]:
# 컬럼명 중 첫번째 행을 기준으로 컬럼 분리
process_cols = [col for col in df.columns if col[0] == 'Process']
sensor_cols = [col for col in df.columns if col[0] == 'Sensor']
defects_cols = [col for col in df.columns if col[0] == 'Defects']

# 데이터 프레임 생성
df_process = df[process_cols].copy()
df_sensor = df[sensor_cols].copy()
df_defects = df[defects_cols].copy()

print("파일 분리")
print(f"Process: 컬럼 {len(process_cols)}개")
print(f"Sensor: 컬럼 {len(sensor_cols)}개")
print(f"Defects: 컬럼 {len(defects_cols)}개")

파일 분리
Process: 컬럼 17개
Sensor: 컬럼 14개
Defects: 컬럼 26개
